*Significant Digits*

In [16]:
def bankers_round_sig_fig(num_str, sig_figs):

    num_str = num_str.strip()
    sign = ""
    if num_str.startswith("-"):
        sign = "-"
        num_str = num_str[1:]

    # Separate integer and fractional parts
    if "." in num_str:
        int_part, frac_part = num_str.split(".")
    else:
        int_part, frac_part = num_str, ""

    full_str = int_part + frac_part

    # Find the first non-zero digit 
    first_nz = -1
    for idx, c in enumerate(full_str):
        if c != '0':
            first_nz = idx
            break

    # If number is zero
    if first_nz == -1:
        return sign + "0"

    # 2. Determine position of 1st significant figure relative to decimal point
    if first_nz < len(int_part):
        first_power = len(int_part) - 1 - first_nz
    else:
        first_power = -(first_nz - len(int_part) + 1)

    target_power = first_power - (sig_figs - 1)
    target_idx = first_nz + sig_figs

    # Pad with extra zeros for safe checking of dropped digits
    full_str = full_str.ljust(target_idx + 10, '0')

    # 3. Analyze dropped digits for Banker's Rounding
    dropped_first = int(full_str[target_idx])
    rest_dropped = full_str[target_idx + 1:]

    if dropped_first > 5:
        go_up = True
    elif dropped_first < 5:
        go_up = False
    else:  # dropped_first == 5
        if any(c != '0' for c in rest_dropped):
            go_up = True
        else:
            # Exactly 0.5: round to nearest even (check last kept digit)
            last_kept = int(full_str[target_idx - 1])
            go_up = (last_kept % 2 != 0)

    # 4. Increment kept digits if rounding up
    kept_digits = full_str[:target_idx]
    val = int(kept_digits)
    if go_up:
        val += 1

    val_str = str(val)

    # 5. Format output string back to standard notation
    if target_power < 0:
        d = -target_power
        val_str = val_str.zfill(d + 1)
        res_int = val_str[:-d]
        res_frac = val_str[-d:]
        result = f"{res_int}.{res_frac}"
    else:
        result = val_str + ("0" * target_power)

    return sign + result


count = int(input("How many numbers? "))
sig_figs = int(input("How many significant figures? "))

numbers = []
for i in range(1, count + 1):
    numbers.append(input(f"Enter number {i}: "))

col2 = "Significant Digits"
print(f"\n{'Number':<16}{col2}")
print("-" * 35)

for num_str in numbers:
    rounded = bankers_round_sig_fig(num_str, sig_figs)
    print(f"{num_str:<16}{rounded}")

How many numbers?  4
How many significant figures?  3
Enter number 1:  2.56778
Enter number 2:  0.0012355
Enter number 3:  2.344500
Enter number 4:  1234.56



Number          Significant Digits
-----------------------------------
2.56778         2.57
0.0012355       0.00124
2.344500        2.34
1234.56         1230


**Banker's Rounding**

In [2]:
def bankers_round(num_str, d):
    num_str = num_str.strip()
    sign = ""
    if num_str.startswith("-"):
        sign = "-"
        num_str = num_str[1:]

    # Separate integer and fractional parts
    if "." in num_str:
        integer_part, frac_part = num_str.split(".")
    else:
        integer_part, frac_part = num_str, ""

    # Ensure fractional part is long enough
    frac_part = frac_part.ljust(d + 10, '0')

    # Dropped digit right after target decimal place 'd'
    first_dropped = int(frac_part[d])
    rest_dropped = frac_part[d + 1:]

    # Determine whether to round up or down
    if first_dropped > 5:
        go_up = True
    elif first_dropped < 5:
        go_up = False
    else: 
        #  first_dropped == 5 Check if there are non-zero digits after 5
        if any(c != '0' for c in rest_dropped):
            go_up = True
        else:
            # Exactly 0.5: check last kept digit
            if d > 0:
                last_kept = int(frac_part[d - 1])
            else:
                last_kept = int(integer_part[-1])

            # If last kept digit is odd -> round up (make even)
            # If last kept digit is even -> stay
            go_up = (last_kept % 2 != 0)

    # Perform increment on the kept digits
    kept_str = integer_part + (frac_part[:d] if d > 0 else "")
    val = int(kept_str)
    if go_up:
        val += 1

    val_str = str(val)

    # Reconstruct integer and fractional parts
    if d > 0:
        val_str = val_str.zfill(len(integer_part) + d)
        res_int = val_str[:-d] if len(val_str) > d else "0"
        res_frac = val_str[-d:]
        result = f"{res_int}.{res_frac}"
    else:
        result = val_str

    return sign + result


count = int(input("How many numbers? "))
d = int(input("How many decimal places? "))

numbers = []
for i in range(1, count + 1):
    val_input = input(f"Enter number {i}: ")
    numbers.append(val_input)

col2 = "Banker's Rounding"
print(f"\n{'Number':<14}{col2}")
print("-" * 31)

for num_str in numbers:
    rounded = bankers_round(num_str, d)
    print(f"{num_str:<14}{rounded}")

How many numbers?  4
How many decimal places?  3
Enter number 1:  2.345600
Enter number 2:  2.344500
Enter number 3:  2.345500
Enter number 4:  7.126500



Number        Banker's Rounding
-------------------------------
2.345600      2.346
2.344500      2.344
2.345500      2.346
7.126500      7.126


**Horner's Method**

In [3]:
degree = int(input("Enter degree of polynomial: "))
coeff = list(map(float, input("Enter coefficients (highest power first): ").split()))

if degree < 0 or len(coeff) != degree + 1:
    print("Invalid degree or coefficient count.")
else:
    x = float(input("Enter value of x: "))
    # Start with the leading coefficient.
    p = coeff[0]
    print("\nPower       Coefficient       P(k)")
    print("----------------------------------")
    print(f"{degree:<12}{coeff[0]:<18g}{p:g}")

    # Horner's recurrence: new P = previous P * x + next coefficient.
    for i in range(1, degree + 1):
        p = p * x + coeff[i]
        power = degree - i
        print(f"{power:<12}{coeff[i]:<18g}{p:g}")

    # The final value is the polynomial evaluated at x.
    print(f"\nf({x:g}) = {p:g}")

Enter degree of polynomial:  4
Enter coefficients (highest power first):  2 -3 0 5 -7
Enter value of x:  2



Power       Coefficient       P(k)
----------------------------------
4           2                 2
3           -3                1
2           0                 2
1           5                 9
0           -7                11

f(2) = 11


** 4. Bisection Method**

In [4]:
# 4. Bisection Method
# Enter polynomial coefficients from highest power to constant term.

def horner(coeff, x):
    # Evaluate the polynomial efficiently using Horner's recurrence.
    result = coeff[0]
    for a in coeff[1:]:
        result = result * x + a
    return result

degree = int(input("Enter degree: "))
coeff = list(map(float, input("Enter coefficients (highest power first): ").split()))
a = float(input("Enter left endpoint a: "))
b = float(input("Enter right endpoint b: "))
tol = float(input("Enter positive tolerance (e.g. 0.0001): "))

if degree < 0 or len(coeff) != degree + 1 or tol <= 0:
    print("Invalid degree, coefficient count, or tolerance.")
else:
    fa, fb = horner(coeff, a), horner(coeff, b)

    # Bisection needs a sign change across the interval, unless an endpoint is a root.
    if fa == 0:
        print("Root =", a)
    elif fb == 0:
        print("Root =", b)
    elif fa * fb > 0:
        print("Invalid interval: f(a) and f(b) must have opposite signs.")
    else:
        print("\nIter        a             b             midpoint       f(midpoint)")
        for i in range(1, 10001):
            # The midpoint divides the current interval into two halves.
            mid = (a + b) / 2
            fm = horner(coeff, mid)
            print(f"{i:<12}{a:<14.7f}{b:<14.7f}{mid:<15.7f}{fm:.7g}")

            # Stop when the function value or interval width is sufficiently small.
            if abs(fm) < tol or (b - a) / 2 < tol:
                print("Approximate root =", mid)
                break

            # Keep the half interval containing the sign change.
            if fa * fm < 0:
                b, fb = mid, fm
            else:
                a, fa = mid, fm
        else:
            print("Maximum iterations reached.")

Enter degree:  3
Enter coefficients (highest power first):   1 0 -1 -2
Enter left endpoint a:  1
Enter right endpoint b:  2
Enter positive tolerance (e.g. 0.0001):  0.001



Iter        a             b             midpoint       f(midpoint)
1           1.0000000     2.0000000     1.5000000      -0.125
2           1.5000000     2.0000000     1.7500000      1.609375
3           1.5000000     1.7500000     1.6250000      0.6660156
4           1.5000000     1.6250000     1.5625000      0.2521973
5           1.5000000     1.5625000     1.5312500      0.05911255
6           1.5000000     1.5312500     1.5156250      -0.0340538
7           1.5156250     1.5312500     1.5234375      0.01225042
8           1.5156250     1.5234375     1.5195312      -0.01097125
9           1.5195312     1.5234375     1.5214844      0.0006221756
Approximate root = 1.521484375


In [ ]:
# Evaluate polynomial using Horner's Rule
def f(coeffs, x):
    result = coeffs[0]
    for c in coeffs[1:]:
        result = result * x + c
    return result

# 1. Dynamic Inputs
n = int(input("Enter degree: "))
coeffs = [float(c) for c in input("Enter coefficients (highest power first): ").split()]
a = float(input("Enter lower limit a: "))
b = float(input("Enter upper limit b: "))
tol = float(input("Enter tolerance: "))

# 2. Check Validity of Interval
if f(coeffs, a) * f(coeffs, b) >= 0:
    print("Invalid interval: f(a) and f(b) must have opposite signs.")
else:
    # 3. Print Table Header
    print(f"\n{'Iteration':<13}{'a':<11}{'b':<11}{'c':<11}{'f(c)'}")
    print("-" * 55)

    iteration = 1
    c = a

    # 4. Bisection Loop
    while (b - a) / 2 > tol:
        c = (a + b) / 2
        fc = f(coeffs, c)

        # Print iteration row
        print(f"{iteration:<13}{a:<11.5g}{b:<11.5g}{c:<11.5g}{fc:.4f}")

        if fc == 0:
            break
        elif f(coeffs, a) * fc < 0:
            b = c
        else:
            a = c

        iteration += 1

    print("-" * 55)
    print(f"Root = {c}")

False methods 

In [5]:
# 5. False Position (Regula Falsi) Method

def horner(coeff, x):
    # Evaluate the polynomial using Horner's recurrence.
    result = coeff[0]
    for a in coeff[1:]:
        result = result * x + a
    return result

degree = int(input("Enter degree: "))
coeff = list(map(float, input("Enter coefficients (highest power first): ").split()))
a = float(input("Enter left endpoint a: "))
b = float(input("Enter right endpoint b: "))
tol = float(input("Enter positive tolerance (e.g. 0.0001): "))

if degree < 0 or len(coeff) != degree + 1 or tol <= 0:
    print("Invalid degree, coefficient count, or tolerance.")
else:
    fa, fb = horner(coeff, a), horner(coeff, b)

    # False position requires opposite signs, unless an endpoint is already a root.
    if fa == 0:
        print("Root =", a)
    elif fb == 0:
        print("Root =", b)
    elif fa * fb > 0:
        print("Invalid interval: f(a) and f(b) must have opposite signs.")
    else:
        print("\nIter        a             b             x             f(x)")
        for i in range(1, 10001):
            denominator = fb - fa
            # Protect against division by zero.
            if denominator == 0:
                print("Cannot continue: f(a) equals f(b).")
                break

            # The x-intercept of the line through (a, f(a)) and (b, f(b)).
            x = (a * fb - b * fa) / denominator
            fx = horner(coeff, x)
            print(f"{i:<12}{a:<14.7f}{b:<14.7f}{x:<14.7f}{fx:.7g}")

            if abs(fx) < tol:
                print("Approximate root =", x)
                break

            # Replace the endpoint that has the same sign as f(x).
            if fa * fx < 0:
                b, fb = x, fx
            else:
                a, fa = x, fx
        else:
            print("Maximum iterations reached.")

Enter degree:  3
Enter coefficients (highest power first):  1 0 -2 -5
Enter left endpoint a:  2
Enter right endpoint b:  3
Enter positive tolerance (e.g. 0.0001):  0.001



Iter        a             b             x             f(x)
1           2.0000000     3.0000000     2.0588235     -0.3907999
2           2.0588235     3.0000000     2.0812637     -0.1472041
3           2.0812637     3.0000000     2.0896392     -0.0546765
4           2.0896392     3.0000000     2.0927396     -0.02020287
5           2.0927396     3.0000000     2.0938837     -0.007450506
6           2.0938837     3.0000000     2.0943055     -0.002745673
7           2.0943055     3.0000000     2.0944608     -0.001011574
8           2.0944608     3.0000000     2.0945181     -0.0003726528
Approximate root = 2.0945180933857572


In [ ]:
# Evaluate polynomial dynamically using Horner's Rule
def f(coeffs, x):
    result = coeffs[0]
    for c in coeffs[1:]:
        result = result * x + c
    return result

# 1. Dynamic Inputs
n = int(input("Enter degree: "))
coeffs = [float(c) for c in input("Enter coefficients (highest power first): ").split()]
a = float(input("Enter lower limit a: "))
b = float(input("Enter upper limit b: "))
E = float(input("Enter error tolerance E: "))

# 2. Check Validity of Initial Interval
fa = f(coeffs, a)
fb = f(coeffs, b)

if fa * fb >= 0:
    print("Invalid interval: f(a) and f(b) must have opposite signs.")
else:
    # 3. Print Slide Table Header
    print(f"\n{'Iteration':<11}{'a':<10}{'b':<10}{'f(a)':<11}{'f(b)':<11}{'x_n':<10}{'f(x_n)':<11}{'Updated Interval'}")
    print("-" * 88)

    iteration = 1
    x_prev = None

    # 4. False Position Iteration Loop
    while True:
        fa = f(coeffs, a)
        fb = f(coeffs, b)

        # Regula Falsi formula
        xn = (a * fb - b * fa) / (fb - fa)
        fxn = f(coeffs, xn)

        # Check stopping condition (converged up to tolerance / decimal match)
        is_converged = (x_prev is not None and abs(xn - x_prev) < E)

        if is_converged:
            updated_interval = "Converged"
            print(f"{iteration:<11}{a:<10.4f}{b:<10.4f}{fa:<11.4f}{fb:<11.4f}{xn:<10.4f}{fxn:<11.4f}{updated_interval}")
            break
        else:
            # Determine new interval bounds
            if fa * fxn < 0:
                next_a, next_b = a, xn
            else:
                next_a, next_b = xn, b

            updated_interval = f"[{next_a:.4f}, {next_b:.4f}]"
            print(f"{iteration:<11}{a:<10.4f}{b:<10.4f}{fa:<11.4f}{fb:<11.4f}{xn:<10.4f}{fxn:<11.4f}{updated_interval}")

            a, b = next_a, next_b
            x_prev = xn
            iteration += 1

    # 5. Final Output Line (matching slide format)
    print()
    print(f"Root ≈ {xn:.4f} (after {iteration} iterations)")

6. Fixed Point Iteration Method


In [6]:
# 6. Fixed Point Iteration Method
import math

# Rewrite f(x) = 0 as x = g(x), then enter the expression for g(x).
# Supported names include x, sin, cos, tan, exp, log, sqrt, pi, and e.
expression = input("Enter g(x), e.g. cos(x) or (x + 2)**(1/3): ")
x = float(input("Enter initial guess x0: "))
tol = float(input("Enter positive tolerance (e.g. 0.0001): "))
max_iter = int(input("Enter maximum iterations: "))

# Provide only a limited set of names to the expression evaluator.
allowed = {
    "x": x, "sin": math.sin, "cos": math.cos, "tan": math.tan,
    "exp": math.exp, "log": math.log, "sqrt": math.sqrt,
    "pi": math.pi, "e": math.e, "abs": abs
}

if tol <= 0 or max_iter <= 0:
    print("Tolerance and maximum iterations must be positive.")
else:
    print("\nIter        x(i)             x(i+1)           error")
    for i in range(1, max_iter + 1):
        try:
            # Calculate the next approximation: x(i+1) = g(x(i)).
            allowed["x"] = x
            next_x = float(eval(expression, {"__builtins__": {}}, allowed))
        except (ValueError, ZeroDivisionError, OverflowError, TypeError, NameError) as error:
            print("Could not evaluate g(x):", error)
            break

        error = abs(next_x - x)
        print(f"{i:<12}{x:<17.9f}{next_x:<17.9f}{error:.7g}")

        # Successive approximations are close enough when error is below tolerance.
        if error < tol:
            print("Approximate fixed point =", next_x)
            break
        x = next_x
    else:
        print("Maximum iterations reached. Last approximation =", x)

Enter g(x), e.g. cos(x) or (x + 2)**(1/3):  sdsdsdsd
Enter initial guess x0:  aessda


ValueError: could not convert string to float: 'aessda'

# 7. Newton-Raphson Method

In [8]:
# Helper to convert float to int if whole number
def parse_val(s):
    val = float(s)
    return int(val) if val.is_integer() else val

# 1. Horner's rule to evaluate polynomial f(x)
def f(coeffs, x):
    result = coeffs[0]
    for c in coeffs[1:]:
        result = result * x + c
    return result

# 2. Function to compute derivative coefficients automatically
def get_derivative_coeffs(coeffs):
    n = len(coeffs) - 1
    d_coeffs = []
    for i in range(n):
        power = n - i
        d_coeffs.append(coeffs[i] * power)
    return d_coeffs

# 3. Inputs matching exam style
n = int(input("Enter degree: "))
coeffs = [float(c) for c in input("Enter coefficients (highest power first): ").split()]
x0_raw = parse_val(input("Enter initial guess x0: "))
x = float(x0_raw)
E = float(input("Enter error tolerance E: "))

# Calculate derivative coefficients automatically
d_coeffs = get_derivative_coeffs(coeffs)

# 4. Table Header
headers = ["n", "x(n)", "f(x(n))", "f'(x(n))", "x(n+1)"]
print(f"\n{headers[0]:<6}{headers[1]:<11}{headers[2]:<11}{headers[3]:<11}{headers[4]}")
print("-" * 52)

step = 1

# 5. Newton-Raphson Iteration Loop
while True:
    fx = f(coeffs, x)
    dfx = f(d_coeffs, x)

    # Division by zero check
    if dfx == 0:
        print("Derivative is zero. Cannot continue.")
        break

    # Newton-Raphson formula: x(n+1) = x(n) - f(x(n)) / f'(x(n))
    x_next = x - fx / dfx

    # Print iteration row formatted to 4 decimal places
    print(f"{step:<6}{x:<11.4f}{fx:<11.4f}{dfx:<11.4f}{x_next:.4f}")

    # Check convergence condition
    if abs(x_next - x) < E:
        break

    x = x_next
    step += 1

# 6. Final Output
print()
print(f"Root = {x_next:.4f}")

Enter degree:  2
Enter coefficients (highest power first):  1 -3 2 
Enter initial guess x0:  0
Enter error tolerance E:  0.0001



n     x(n)       f(x(n))    f'(x(n))   x(n+1)
----------------------------------------------------
1     0.0000     2.0000     -3.0000    0.6667
2     0.6667     0.4444     -1.6667    0.9333
3     0.9333     0.0711     -1.1333    0.9961
4     0.9961     0.0039     -1.0078    1.0000
5     1.0000     0.0000     -1.0000    1.0000

Root = 1.0000


In [7]:
# 7. Newton-Raphson Method

def horner(coeff, x):
    # Evaluate a polynomial using Horner's recurrence.
    result = coeff[0]
    for a in coeff[1:]:
        result = result * x + a
    return result

degree = int(input("Enter degree: "))
coeff = list(map(float, input("Enter coefficients (highest power first): ").split()))
x = float(input("Enter initial guess x0: "))
tol = float(input("Enter positive tolerance (e.g. 0.0001): "))
max_iter = int(input("Enter maximum iterations: "))

if degree < 0 or len(coeff) != degree + 1 or tol <= 0 or max_iter <= 0:
    print("Invalid degree, coefficient count, tolerance, or iteration limit.")
else:
    # Differentiate each polynomial term to create derivative coefficients.
    derivative = [coeff[i] * (degree - i) for i in range(degree)]

    if not derivative:
        print("A constant polynomial cannot use Newton-Raphson iteration.")
    else:
        print("\nIter        x(i)             f(x(i))          x(i+1)")
        for i in range(1, max_iter + 1):
            fx = horner(coeff, x)
            dfx = horner(derivative, x)

            # The method divides by f'(x), so a zero derivative stops iteration.
            if dfx == 0:
                print("Derivative is zero. Cannot continue.")
                break

            # Newton-Raphson formula: x(next) = x - f(x) / f'(x).
            next_x = x - fx / dfx
            print(f"{i:<12}{x:<17.9f}{fx:<17.9g}{next_x:.9f}")

            if abs(next_x - x) < tol or abs(fx) < tol:
                print("Approximate root =", next_x)
                break
            x = next_x
        else:
            print("Maximum iterations reached. Last approximation =", x)

Enter degree:  2
Enter coefficients (highest power first):  1 -3 2
Enter initial guess x0:  0
Enter positive tolerance (e.g. 0.0001):  0.0001
Enter maximum iterations:  8



Iter        x(i)             f(x(i))          x(i+1)
1           0.000000000      2                0.666666667
2           0.666666667      0.444444444      0.933333333
3           0.933333333      0.0711111111     0.996078431
4           0.996078431      0.00393694733    0.999984741
5           0.999984741      1.52592547e-05   1.000000000
Approximate root = 0.9999999997671692


**secant_method**

In [9]:

# Q2: Find a polynomial root using the Secant Method

# Function to evaluate a polynomial using Horner's rule
def horner(coeff, x):
    # Start with the leading coefficient
    p = coeff[0]

    # Evaluate the polynomial without using powers explicitly
    for i in range(1, len(coeff)):
        p = p * x + coeff[i]

    # Return the polynomial value f(x)
    return p


# Read the degree of the polynomial
n = int(input("Enter degree: "))

# Read coefficients from highest power to lowest power
coeff = list(map(float, input(
    "Enter coefficients (highest power first): "
).split()))

# Read the two initial guesses
x0, x1 = map(float, input("Enter x0 and x1: ").split())

# Read the required number of decimal places
d = int(input("Correct up to how many decimal places? "))

# Evaluate the polynomial at the initial guesses
f0 = horner(coeff, x0)
f1 = horner(coeff, x1)

# The initial guesses must have opposite signs of f(x)
if f0 * f1 >= 0:
    print("Invalid initial values")
else:
    # Print the iteration table heading
    print("\ni    x(i)       x(i-1)     f(x(i))"
          "     f(x(i-1))   x(i+1)")
    print("-" * 75)

    # Set the iteration counter
    i = 1

    # Continue until successive roots agree to d decimal places
    while True:
        # Evaluate the denominator of the Secant formula
        denominator = f1 - f0

        # Avoid division by zero
        if denominator == 0:
            print("Division by zero")
            break

        # Calculate the next approximation using the Secant formula
        x2 = x1 - f1 * (x1 - x0) / denominator

        # Print the current iteration and its function values
        print(f"{i:<5}{x1:<11.{d + 3}f}"
              f"{x0:<11.{d + 3}f}"
              f"{f1:<13.{d + 3}f}"
              f"{f0:<13.{d + 3}f}"
              f"{x2:.{d + 3}f}")

        # Stop if the current and next approximations
        # are equal when rounded to the required decimal places
        if round(x1, d) == round(x2, d):
            print(f"\nRoot = {x2:.{d}f}")
            break

        # Shift the approximations for the next iteration
        x0 = x1
        f0 = f1

        x1 = x2
        f1 = horner(coeff, x1)

        # Increase the iteration counter
        i += 1

Enter degree:  3
Enter coefficients (highest power first):  2 0 -2 -5
Enter x0 and x1:  1 2 
Correct up to how many decimal places?  2



i    x(i)       x(i-1)     f(x(i))     f(x(i-1))   x(i+1)
---------------------------------------------------------------------------
1    2.00000    1.00000    7.00000      -5.00000     1.41667
2    1.41667    2.00000    -2.14699     7.00000      1.55359
3    1.55359    1.41667    -0.60759     -2.14699     1.60763
4    1.60763    1.55359    0.09449      -0.60759     1.60036
5    1.60036    1.60763    -0.00324     0.09449      1.60060

Root = 1.60


 9. Newton's Forward

In [10]:
#Newton Forward Interpolation
# Function to convert floating number to integer format if whole number
def parse_val(s):
    val = float(s)
    return int(val) if val.is_integer() else val

# 1. Inputs matching exact exam format
n = int(input("Enter number of points: "))
x = [float(v) for v in input("Enter x values: ").split()]
y = [float(v) for v in input("Enter y values: ").split()]
target_x_raw = parse_val(input("Enter x to find f(x): "))
target_x = float(target_x_raw)

# 2. 2D Forward Difference Table Calculation
diff = [[0.0] * n for _ in range(n)]

for i in range(n):
    diff[i][0] = y[i]

for j in range(1, n):
    for i in range(n - j):
        diff[i][j] = diff[i + 1][j - 1] - diff[i][j - 1]

# 3. Table Headers
headers = ["x", "Y"]
for j in range(1, n):
    if j == 1:
        headers.append("1st Diff")
    elif j == 2:
        headers.append("2nd Diff")
    elif j == 3:
        headers.append("3rd Diff")
    else:
        headers.append(f"{j}th Diff")

# Print Header
header_str = f"{headers[0]:<10}{headers[1]:<10}"
for h in headers[2:]:
    header_str += f"{h:<11}"

print(header_str.rstrip())
print("-" * len(header_str.rstrip()))

# Print Table Rows
for i in range(n):
    row_str = f"{x[i]:<10.4f}{y[i]:<10.4f}"
    for j in range(1, n - i):
        row_str += f"{diff[i][j]:<11.4f}"
    print(row_str.rstrip())

# 4. Newton's Forward Interpolation Calculation
h = x[1] - x[0]
u = (target_x - x[0]) / h

result = y[0]
u_term = 1.0

for j in range(1, n):
    u_term *= (u - j + 1) / j
    result += u_term * diff[0][j]

# 5. Output Result formatted to 4 decimal places
print()
print(f"f({target_x_raw}) = {result:.4f}")

Enter number of points:  5
Enter x values:  1891 1901 1911 1921 1931
Enter y values:  46 66 81 93 101
Enter x to find f(x):  1895


x         Y         1st Diff   2nd Diff   3rd Diff   4th Diff
-------------------------------------------------------------
1891.0000 46.0000   20.0000    -5.0000    2.0000     -3.0000
1901.0000 66.0000   15.0000    -3.0000    -1.0000
1911.0000 81.0000   12.0000    -4.0000
1921.0000 93.0000   8.0000
1931.0000 101.0000

f(1895) = 54.8528


9. # Newton Backward Interpolation

In [11]:
# Function to parse string number as int if whole, else float
def parse_val(s):
    val = float(s)
    return int(val) if val.is_integer() else val

# 1. Inputs matching exact prompt format
n = int(input("Enter number of points: "))
x = [float(v) for v in input("Enter x values: ").split()]
y = [float(v) for v in input("Enter y values: ").split()]
target_x_raw = parse_val(input("Enter x to find f(x): "))
target_x = float(target_x_raw)

# 2. Construct 2D Backward Difference Table
diff = [[0.0] * n for _ in range(n)]

for i in range(n):
    diff[i][0] = y[i]

for j in range(1, n):
    for i in range(j, n):
        diff[i][j] = diff[i][j - 1] - diff[i - 1][j - 1]

# 3. Dynamic Column Headers
headers = ["x", "Y"]
for j in range(1, n):
    if j == 1:
        headers.append("1st Diff")
    elif j == 2:
        headers.append("2nd Diff")
    elif j == 3:
        headers.append("3rd Diff")
    else:
        headers.append(f"{j}th Diff")

# Print Table Header
header_str = f"{headers[0]:<10}{headers[1]:<10}"
for h in headers[2:]:
    header_str += f"{h:<11}"

print(header_str.rstrip())
print("-" * len(header_str.rstrip()))

# Print Row by Row
for i in range(n):
    row_str = f"{x[i]:<10.4f}{y[i]:<10.4f}"
    for j in range(1, i + 1):
        row_str += f"{diff[i][j]:<11.4f}"
    print(row_str.rstrip())

# 4. Calculate Newton's Backward Interpolation Formula
h = x[1] - x[0]
u = (target_x - x[n - 1]) / h

result = y[n - 1]
u_term = 1.0

for j in range(1, n):
    u_term *= (u + j - 1) / j
    result += u_term * diff[n - 1][j]

# 5. Output Result formatted to 4 decimal places
print()
print(f"f({target_x_raw}) = {result:.4f}")

Enter number of points:  5
Enter x values:  0 1 2 3 4
Enter y values:  1 3 11 31 69
Enter x to find f(x):  3.5


x         Y         1st Diff   2nd Diff   3rd Diff   4th Diff
-------------------------------------------------------------
0.0000    1.0000
1.0000    3.0000    2.0000
2.0000    11.0000   8.0000     6.0000
3.0000    31.0000   20.0000    12.0000    6.0000
4.0000    69.0000   38.0000    18.0000    6.0000     0.0000

f(3.5) = 47.3750


# 10. Newton's Divided Difference Interpolation

In [14]:
# Helper to format numbers cleanly
def parse_val(s):
    val = float(s)
    return int(val) if val.is_integer() else val

# 1. Dynamic Inputs matching exam standard
n = int(input("Enter number of points: "))
x = [float(v) for v in input("Enter x values: ").split()]
y = [float(v) for v in input("Enter y values: ").split()]
target_x_raw = parse_val(input("Enter x to find f(x): "))
target_x = float(target_x_raw)

# 2. Construct Divided Difference Table
diff = [[0.0] * n for _ in range(n)]

for i in range(n):
    diff[i][0] = y[i]

for j in range(1, n):
    for i in range(n - j):
        diff[i][j] = (diff[i + 1][j - 1] - diff[i][j - 1]) / (x[i + j] - x[i])

# 3. Dynamic Column Headers
headers = ["X", "f(x)"]
for j in range(1, n):
    if j == 1:
        headers.append("1st Diff")
    elif j == 2:
        headers.append("2nd Diff")
    elif j == 3:
        headers.append("3rd Diff")
    else:
        headers.append(f"{j}th Diff")

# Print Table Header
header_str = f"{headers[0]:<10}{headers[1]:<10}"
for h in headers[2:]:
    header_str += f"{h:<11}"

print(header_str.rstrip())
print("-" * len(header_str.rstrip()))

# Print Table Rows
for i in range(n):
    row_str = f"{x[i]:<10.4f}{y[i]:<10.4f}"
    for j in range(1, n - i):
        row_str += f"{diff[i][j]:<11.4f}"
    print(row_str.rstrip())

# 4. Calculate Interpolation using Divided Difference Formula
result = diff[0][0]
term = 1.0

for j in range(1, n):
    term *= (target_x - x[j - 1])
    result += term * diff[0][j]

# 5. Output Result formatted to 4 decimal places
print()
print(f"f({target_x_raw}) = {result:.4f}")

Enter number of points:  5
Enter x values:  5 7 11 13 21
Enter y values:  150 392 1452 2366 9702
Enter x to find f(x):  6


X         f(x)      1st Diff   2nd Diff   3rd Diff   4th Diff
-------------------------------------------------------------
5.0000    150.0000  121.0000   24.0000    1.0000     0.0000
7.0000    392.0000  265.0000   32.0000    1.0000
11.0000   1452.0000 457.0000   46.0000
13.0000   2366.0000 917.0000
21.0000   9702.0000

f(6) = 252.0000


both

In [15]:
# 9. Newton's Forward and Backward Interpolation

import math

def difference_table(y):
    # Build successive difference columns: y, first differences, second differences...
    table = [list(y)]
    while len(table[-1]) > 1:
        previous = table[-1]
        table.append([
            previous[i + 1] - previous[i]
            for i in range(len(previous) - 1)
        ])
    return table

n = int(input("Enter number of data points: "))
x_values = list(map(float, input("Enter x values in order: ").split()))
y_values = list(map(float, input("Enter corresponding y values: ").split()))

if n < 2 or len(x_values) != n or len(y_values) != n:
    print("Enter at least two points and exactly n x and y values.")
else:
    h = x_values[1] - x_values[0]

    # Forward/backward interpolation assumes equally spaced x values.
    equally_spaced = h != 0 and all(
        math.isclose(x_values[i + 1] - x_values[i], h,
                     rel_tol=1e-9, abs_tol=1e-9)
        for i in range(1, n - 1)
    )

    if not equally_spaced:
        print("x values must be distinct and equally spaced.")
    else:
        target = float(input("Enter interpolation value: "))
        table = difference_table(y_values)

        # Print the triangular difference table.
        print("\nDifference table:")
        for i in range(n):
            row = [
                f"{table[j][i]:.6g}" if i < len(table[j]) else ""
                for j in range(n)
            ]
            print(f"{i:<5}" + "".join(f"{value:>14}" for value in row))

        # Choose forward near the start; backward near the end.
        if abs((target - x_values[0]) / h) <= abs((target - x_values[-1]) / h):
            u = (target - x_values[0]) / h
            result = y_values[0]
            term = 1.0

            # Forward formula uses Δy0, Δ²y0, etc.
            for order in range(1, n):
                term *= (u - (order - 1)) / order
                result += term * table[order][0]
            print("Method: Newton forward interpolation")
        else:
            u = (target - x_values[-1]) / h
            result = y_values[-1]
            term = 1.0

            # Backward formula uses the differences at the bottom of each column.
            for order in range(1, n):
                term *= (u + (order - 1)) / order
                result += term * table[order][n - order - 1]
            print("Method: Newton backward interpolation")

        print(f"Interpolated value at x = {target:g} is {result:.10g}")

Enter number of data points:  5
Enter x values in order:  1891 1901 1911 1921 1931
Enter corresponding y values:   46 66 81 93 101
Enter interpolation value:  1895



Difference table:
0                46            20            -5             2            -3
1                66            15            -3            -1              
2                81            12            -4                            
3                93             8                                          
4               101                                                        
Method: Newton forward interpolation
Interpolated value at x = 1895 is 54.8528
